# Session 22 Code-Along: Policy Trees in Python

**HPM 883 — Advanced Quantitative Methods**
*From CATEs to Treatment Rules — Spring 2026*

---

This notebook walks through the full policy learning pipeline on a synthetic mobile-health adherence dataset. You'll estimate CATEs, build doubly-robust policy scores, fit a depth-2 policy tree, and see what happens under a budget constraint.

**Goals for today:**

1. Generate a synthetic dataset with known optimal policy — so we can check our work
2. Estimate CATEs with a causal forest (T-learner with honesty)
3. Build per-action doubly-robust scores
4. Fit a depth-2 policy tree and visualize it
5. Compare four policies: treat-all, treat-none, naive threshold, and the policy tree
6. See a failure case (constant effects → spurious splits)
7. Add a 30% budget constraint and see how the rule simplifies

**How to use this notebook in class:** Run each cell as we walk through it. Tinker with the knobs at the top. If you get stuck, the code is self-contained — you can restart and re-run from the top in under a minute.

## Setup

Nothing fancy. Pure `numpy`, `pandas`, `scikit-learn`, and `matplotlib`. No extra installs needed — Colab has all of these preinstalled.

If you're running locally and something is missing, uncomment the `%pip install` line.

In [ ]:
# %pip install -q numpy pandas scikit-learn matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import KFold

RNG = np.random.default_rng(883)  # course seed
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11
print("Setup complete.")

## 1. Generate a Synthetic mHealth Dataset

**The scenario.** A health system is piloting a mobile adherence app for type-2 diabetes patients. The app sends reminders and tracks medication, glucose, and activity. You've run a randomized trial (n = 2000) and want to learn: **who should be offered the app?**

**The data generating process (ground truth — we only get to see this because it's synthetic):**

- $X_1$: age (years, 30–80)
- $X_2$: baseline A1C (continuous, 6.0–11.0)
- $X_3$: digital literacy score (0–1)
- $X_4$: distance to clinic (km, 0–50)
- $X_5$: comorbidity count (0–5)

**True CATE:**
$$\tau(x) = 0.8 \cdot \mathbb{1}\{\text{age} > 50\} \cdot \mathbb{1}\{\text{digital\_literacy} > 0.5\} - 0.3 \cdot \mathbb{1}\{\text{A1C} < 7\} + 0.2$$

In English: older patients with decent digital skills benefit a lot. Well-controlled patients (A1C < 7) are actually *hurt* slightly (false alarms, disengagement). Everyone else gets a small benefit of 0.2.

**Optimal policy** (if we knew $\tau$):

$$\pi^*(x) = \begin{cases} \text{Treat} & \text{if }(\text{age} > 50\text{ AND digital\_literacy} > 0.5)\text{ AND NOT }(\text{A1C} < 7) \\ \text{Don't treat} & \text{otherwise if }\tau(x) \leq 0 \\ \text{Treat} & \text{otherwise (small base benefit)} \end{cases}$$

Let's simulate it.

In [ ]:
def generate_data(n=2000, rng=None):
    if rng is None:
        rng = np.random.default_rng(883)

    # Covariates
    age = rng.uniform(30, 80, n)
    a1c = rng.uniform(6.0, 11.0, n)
    digital_literacy = rng.uniform(0, 1, n)
    distance = rng.uniform(0, 50, n)
    comorbidity = rng.integers(0, 6, n)

    X = np.column_stack([age, a1c, digital_literacy, distance, comorbidity])
    feature_names = ["age", "a1c", "digital_literacy", "distance", "comorbidity"]

    # True CATE (ground truth — hidden from the learner)
    tau = (
        0.8 * ((age > 50) & (digital_literacy > 0.5)).astype(float)
        - 0.3 * (a1c < 7).astype(float)
        + 0.2
    )

    # Baseline outcome (adherence score, 0-10 scale)
    mu0 = (
        5.0
        - 0.02 * (age - 50)
        - 0.3 * (a1c - 8)
        + 0.5 * digital_literacy
        - 0.01 * distance
        - 0.2 * comorbidity
        + rng.normal(0, 0.5, n)
    )
    mu1 = mu0 + tau  # potential outcome under treatment

    # RCT: randomize treatment with probability 0.5
    W = rng.binomial(1, 0.5, n)
    Y = np.where(W == 1, mu1, mu0)  # observed outcome

    return pd.DataFrame({
        "age": age, "a1c": a1c, "digital_literacy": digital_literacy,
        "distance": distance, "comorbidity": comorbidity,
        "W": W, "Y": Y, "tau_true": tau
    }), X, feature_names

df, X, feature_names = generate_data(n=2000, rng=RNG)
print(f"n = {len(df)},  treated fraction = {df['W'].mean():.2f}")
print(f"True ATE = {df['tau_true'].mean():.3f}")
print(f"True optimal treated fraction = {(df['tau_true'] > 0).mean():.2f}")
df.head()

### Visualize the True CATE

Before we do any estimation, let's peek at the ground truth. This is cheating — in the real world you never see the true $\tau(x)$. But for a teaching demo, we want to know what the right answer looks like.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel 1: Overall CATE distribution
axes[0].hist(df["tau_true"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(0, color="black", linestyle="--", linewidth=1, label="τ(x) = 0")
axes[0].axvline(df["tau_true"].mean(), color="red", linestyle="-", linewidth=1.5,
                label=f"ATE = {df['tau_true'].mean():.2f}")
axes[0].set_xlabel("True CATE τ(x)")
axes[0].set_ylabel("Patients")
axes[0].set_title("True CATE distribution")
axes[0].legend()

# Panel 2: CATE by subgroup
subgroups = {
    "age>50 ∧ digital>0.5 ∧ A1C≥7": (df["age"] > 50) & (df["digital_literacy"] > 0.5) & (df["a1c"] >= 7),
    "age>50 ∧ digital>0.5 ∧ A1C<7": (df["age"] > 50) & (df["digital_literacy"] > 0.5) & (df["a1c"] < 7),
    "A1C<7 only": (df["a1c"] < 7) & ~((df["age"] > 50) & (df["digital_literacy"] > 0.5)),
    "Other": ~((df["age"] > 50) & (df["digital_literacy"] > 0.5)) & (df["a1c"] >= 7),
}
means = [df.loc[mask, "tau_true"].mean() for mask in subgroups.values()]
colors = ["darkgreen", "orange", "darkred", "gray"]
bars = axes[1].bar(range(len(subgroups)), means, color=colors, edgecolor="white")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_xticks(range(len(subgroups)))
axes[1].set_xticklabels(list(subgroups.keys()), rotation=20, ha="right", fontsize=9)
axes[1].set_ylabel("Mean true CATE")
axes[1].set_title("True CATE by subgroup")
for bar, m in zip(bars, means):
    axes[1].annotate(f"{m:.2f}", (bar.get_x() + bar.get_width()/2, m),
                     ha="center", va="bottom" if m > 0 else "top", fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 The target: the green bar is the only subgroup where treatment clearly helps. The orange bar is the trap — positive digital-literacy BUT well-controlled A1C, where the benefit is small or negative.")

## 2. Estimate CATEs (The Honest Way)

We don't get to see `tau_true` in a real study. We have to estimate $\tau(x)$ from $(Y, W, X)$. We'll use a **T-learner**: fit separate outcome models for treated and control, then take the difference.

The sklearn-native approach:

1. Fit $\hat\mu_0(x)$ on the control subsample
2. Fit $\hat\mu_1(x)$ on the treated subsample
3. For each person, predict both counterfactuals: $\hat\tau(x) = \hat\mu_1(x) - \hat\mu_0(x)$

To avoid in-sample overfitting bias (the honesty principle from Unit 3), we use **K-fold cross-fitting**: for each fold, train on the other folds and predict on the held-out fold.

In [ ]:
def t_learner_crossfit(X, W, Y, n_folds=5, rng=None):
    """Cross-fit T-learner: returns out-of-fold mu0_hat, mu1_hat, and tau_hat."""
    if rng is None:
        rng = np.random.default_rng(883)
    n = len(Y)
    mu0_hat = np.zeros(n)
    mu1_hat = np.zeros(n)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=883)
    for train_idx, test_idx in kf.split(X):
        X_tr, W_tr, Y_tr = X[train_idx], W[train_idx], Y[train_idx]
        X_te = X[test_idx]

        # Outcome model for control arm
        rf0 = RandomForestRegressor(n_estimators=300, min_samples_leaf=10, random_state=883, n_jobs=-1)
        rf0.fit(X_tr[W_tr == 0], Y_tr[W_tr == 0])

        # Outcome model for treated arm
        rf1 = RandomForestRegressor(n_estimators=300, min_samples_leaf=10, random_state=883, n_jobs=-1)
        rf1.fit(X_tr[W_tr == 1], Y_tr[W_tr == 1])

        mu0_hat[test_idx] = rf0.predict(X_te)
        mu1_hat[test_idx] = rf1.predict(X_te)

    tau_hat = mu1_hat - mu0_hat
    return mu0_hat, mu1_hat, tau_hat

mu0_hat, mu1_hat, tau_hat = t_learner_crossfit(X, df["W"].values, df["Y"].values, n_folds=5)

print(f"Estimated ATE: {tau_hat.mean():.3f}")
print(f"True ATE:      {df['tau_true'].mean():.3f}")
print(f"Corr(tau_hat, tau_true) = {np.corrcoef(tau_hat, df['tau_true'])[0, 1]:.3f}")

### Sanity Check: Does $\hat\tau$ Recover $\tau_{\text{true}}$?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Panel 1: scatter
axes[0].scatter(df["tau_true"], tau_hat, alpha=0.3, s=10, color="steelblue")
axes[0].plot([-0.5, 1.2], [-0.5, 1.2], "k--", linewidth=1, label="45° line")
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].axvline(0, color="gray", linewidth=0.5)
axes[0].set_xlabel("True τ(x)")
axes[0].set_ylabel("Estimated τ̂(x)")
axes[0].set_title(f"CATE recovery (r = {np.corrcoef(tau_hat, df['tau_true'])[0, 1]:.2f})")
axes[0].legend()

# Panel 2: histograms
axes[1].hist(df["tau_true"], bins=40, alpha=0.5, label="True τ(x)", color="darkgreen")
axes[1].hist(tau_hat, bins=40, alpha=0.5, label="Estimated τ̂(x)", color="orange")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Treatment effect")
axes[1].set_ylabel("Patients")
axes[1].set_title("True vs. estimated CATE distributions")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nThe T-learner captures the overall shape but is noisier than the ground truth — exactly what you'd expect from any finite-sample estimator. The correlation tells you how much useful signal is in τ̂.")

## 3. Doubly-Robust Policy Scores

Here's the move that makes policy learning work. Instead of deciding who to treat based on $\hat\tau > 0$, we construct **per-action doubly-robust scores**:

$$\Gamma_{i,0} = \hat\mu_0(X_i) + \frac{\mathbb{1}\{W_i = 0\}}{1-\hat e(X_i)}\bigl(Y_i - \hat\mu_0(X_i)\bigr)$$

$$\Gamma_{i,1} = \hat\mu_1(X_i) + \frac{\mathbb{1}\{W_i = 1\}}{\hat e(X_i)}\bigl(Y_i - \hat\mu_1(X_i)\bigr)$$

Each $\Gamma_{i,a}$ is an unbiased estimate of $Y_i(a)$. The genius is that we can now evaluate **any policy** by looking up $\Gamma_{i,\pi(X_i)}$ for each person and averaging — without ever re-fitting the outcome models.

In an RCT with balanced assignment, $\hat e(x) = 0.5$ for everyone.

In [ ]:
e_hat = np.full(len(df), 0.5)  # RCT: treatment probability = 0.5 for everyone
W = df["W"].values
Y = df["Y"].values

# Per-action DR scores
gamma_0 = mu0_hat + ((W == 0).astype(float) / (1 - e_hat)) * (Y - mu0_hat)
gamma_1 = mu1_hat + ((W == 1).astype(float) / e_hat) * (Y - mu1_hat)

# Sanity: mean of gamma_1 - gamma_0 should ≈ ATE
print(f"Mean(Γ_1 - Γ_0) = {(gamma_1 - gamma_0).mean():.3f}  (DR estimate of ATE)")
print(f"True ATE        = {df['tau_true'].mean():.3f}")
print(f"T-learner ATE   = {tau_hat.mean():.3f}")
print(f"\nGamma_0 range: [{gamma_0.min():.2f}, {gamma_0.max():.2f}]")
print(f"Gamma_1 range: [{gamma_1.min():.2f}, {gamma_1.max():.2f}]")

### Why This Is the Right Input

With DR scores in hand, the **value** of any policy $\pi$ is simply:

$$\hat V(\pi) = \frac{1}{n}\sum_i \Gamma_{i,\pi(X_i)}$$

We're going to use this to score four policies.

In [ ]:
def policy_value(pi_assignments, gamma_0, gamma_1):
    """Compute DR value of a policy given per-action scores.
    pi_assignments: 0/1 array (0 = don't treat, 1 = treat)"""
    chosen_scores = np.where(pi_assignments == 1, gamma_1, gamma_0)
    val = chosen_scores.mean()
    se = chosen_scores.std(ddof=1) / np.sqrt(len(chosen_scores))
    return val, se

# Four policies
pi_treat_all  = np.ones(len(df), dtype=int)
pi_treat_none = np.zeros(len(df), dtype=int)
pi_naive      = (tau_hat > 0).astype(int)  # naive threshold

# For comparison, the oracle (only possible in sim)
pi_oracle     = (df["tau_true"] > 0).astype(int).values

results = []
for name, pi in [("Treat all", pi_treat_all),
                 ("Treat none", pi_treat_none),
                 ("Naive (τ̂ > 0)", pi_naive),
                 ("Oracle (τ_true > 0)", pi_oracle)]:
    v, se = policy_value(pi, gamma_0, gamma_1)
    results.append({"Policy": name, "Treated frac": pi.mean(),
                    "Value": v, "SE": se, "CI_low": v - 1.96*se, "CI_high": v + 1.96*se})

results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

## 4. The Policy Tree: Interpretable Targeting

Now the centerpiece. We want a **shallow decision tree** (depth 2) that tells a program manager *exactly* who to treat, using at most 3 splits.

**The trick.** A policy tree is just a classifier trained to predict the **argmax action** under DR scores — weighted by how much that action wins.

More precisely: at each person, the "right" action is $\arg\max_a \Gamma_{i,a}$, and the sample importance of that decision is $|\Gamma_{i,1} - \Gamma_{i,0}|$. We fit `DecisionTreeClassifier` with these weights. This is the same objective `policytree::policy_tree` in R optimizes (minus the exact-search machinery — `DecisionTreeClassifier` uses greedy splits, which are near-optimal for depth 2).

In [ ]:
def fit_policy_tree(X, gamma_0, gamma_1, max_depth=2, feature_names=None):
    """Fit a depth-limited policy tree from per-action DR scores."""
    # Target: which action wins
    best_action = (gamma_1 > gamma_0).astype(int)
    # Weight: how much it wins by
    sample_weight = np.abs(gamma_1 - gamma_0)

    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        criterion="gini",
        min_samples_leaf=50,
        random_state=883
    )
    tree.fit(X, best_action, sample_weight=sample_weight)
    return tree

tree2 = fit_policy_tree(X, gamma_0, gamma_1, max_depth=2, feature_names=feature_names)

# Visualize
fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree2, feature_names=feature_names, class_names=["Don't treat", "Treat"],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title("Depth-2 Policy Tree", fontsize=13)
plt.tight_layout()
plt.show()

pi_tree2 = tree2.predict(X)
v_tree2, se_tree2 = policy_value(pi_tree2, gamma_0, gamma_1)
print(f"\nDepth-2 tree treated fraction: {pi_tree2.mean():.2%}")
print(f"Depth-2 tree policy value:     {v_tree2:.3f}  (SE {se_tree2:.3f})")

### Think About It

Look at the tree above. **Did it find the true structure?** Specifically:

- Does it split on **age** or **digital_literacy** in the root? Both matter by construction.
- Does it catch the **A1C < 7 harm subgroup**? A depth-2 tree can only split on 3 variables — it might drop the harm term to keep the benefit term.
- Where would the tree be *wrong*? Spot the misclassified regions in the leaves.

This is the **interpretability–optimality tradeoff** in action. The tree gives up some value relative to the oracle but gives you something you can explain on a napkin.

## 5. Compare All Policies Side-by-Side

In [ ]:
# Add depth-2 and depth-3 trees to comparison
tree3 = fit_policy_tree(X, gamma_0, gamma_1, max_depth=3, feature_names=feature_names)
pi_tree3 = tree3.predict(X)
v_tree3, se_tree3 = policy_value(pi_tree3, gamma_0, gamma_1)

all_results = results + [
    {"Policy": "Policy tree (d=2)", "Treated frac": pi_tree2.mean(),
     "Value": v_tree2, "SE": se_tree2, "CI_low": v_tree2 - 1.96*se_tree2, "CI_high": v_tree2 + 1.96*se_tree2},
    {"Policy": "Policy tree (d=3)", "Treated frac": pi_tree3.mean(),
     "Value": v_tree3, "SE": se_tree3, "CI_low": v_tree3 - 1.96*se_tree3, "CI_high": v_tree3 + 1.96*se_tree3},
]
all_df = pd.DataFrame(all_results)

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
ordering = all_df.sort_values("Value").reset_index(drop=True)
colors = ["#999999", "#cccccc", "#4477AA", "#228833", "#EE6677", "#CCBB44"]
bars = ax.barh(range(len(ordering)), ordering["Value"], color=colors,
               xerr=1.96 * ordering["SE"], capsize=4)
ax.set_yticks(range(len(ordering)))
ax.set_yticklabels(ordering["Policy"])
ax.set_xlabel("Policy value (DR estimate, 95% CI)")
ax.set_title("Policy comparison on St. Null's data (synthetic)")
ax.grid(axis="x", alpha=0.3)
for i, (v, frac) in enumerate(zip(ordering["Value"], ordering["Treated frac"])):
    ax.annotate(f"  {v:.2f}  ({frac:.0%} treated)", (v, i), va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\n📈 Key observations:")
print(f"  • Oracle achieves the best value — it's the ceiling")
print(f"  • Depth-3 tree beats depth-2 (more expressivity, closer to oracle)")
print(f"  • Both trees beat naive (τ̂ > 0) — the tree's structure helps regularize against estimation noise")
print(f"  • Treat-all is suboptimal because some subgroups are actually hurt")

## 6. Failure Case: Constant Treatment Effect

What happens if there's **no heterogeneity** at all? Every patient gets the same benefit. A good policy learner should say "treat everyone" — but a greedy tree might invent splits to overfit to DR-score noise.

Let's check.

In [ ]:
# Generate new data with a CONSTANT effect (no heterogeneity)
def generate_constant_effect_data(n=2000, rng=None):
    if rng is None:
        rng = np.random.default_rng(1234)
    age = rng.uniform(30, 80, n)
    a1c = rng.uniform(6.0, 11.0, n)
    digital_literacy = rng.uniform(0, 1, n)
    distance = rng.uniform(0, 50, n)
    comorbidity = rng.integers(0, 6, n)
    X = np.column_stack([age, a1c, digital_literacy, distance, comorbidity])

    tau = np.full(n, 0.25)  # CONSTANT — no heterogeneity
    mu0 = 5.0 - 0.02*(age-50) - 0.3*(a1c-8) + 0.5*digital_literacy - 0.01*distance - 0.2*comorbidity + rng.normal(0, 0.5, n)
    mu1 = mu0 + tau

    W = rng.binomial(1, 0.5, n)
    Y = np.where(W == 1, mu1, mu0)
    return pd.DataFrame({"age":age,"a1c":a1c,"digital_literacy":digital_literacy,
                          "distance":distance,"comorbidity":comorbidity,
                          "W":W,"Y":Y,"tau_true":tau}), X

df_const, X_const = generate_constant_effect_data(n=2000)

# Estimate CATEs and DR scores on this data
mu0_c, mu1_c, tau_c = t_learner_crossfit(X_const, df_const["W"].values, df_const["Y"].values, n_folds=5)
e_c = np.full(len(df_const), 0.5)
gamma_0_c = mu0_c + ((df_const["W"]==0).values.astype(float)/(1-e_c)) * (df_const["Y"].values - mu0_c)
gamma_1_c = mu1_c + ((df_const["W"]==1).values.astype(float)/e_c) * (df_const["Y"].values - mu1_c)

# Fit policy tree
tree_const = fit_policy_tree(X_const, gamma_0_c, gamma_1_c, max_depth=2, feature_names=feature_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(tau_c, bins=40, color="crimson", edgecolor="white")
axes[0].axvline(0.25, color="black", linestyle="--", label="True τ = 0.25")
axes[0].set_xlabel("Estimated τ̂(x)")
axes[0].set_ylabel("Patients")
axes[0].set_title("CATE estimates when the truth is CONSTANT")
axes[0].legend()

plot_tree(tree_const, feature_names=feature_names, class_names=["Don't","Treat"],
          filled=True, rounded=True, fontsize=9, ax=axes[1])
axes[1].set_title("Policy tree on data with NO real heterogeneity")
plt.tight_layout()
plt.show()

frac_treated_const = tree_const.predict(X_const).mean()
print(f"\n⚠️  The policy tree is still splitting on covariates — there's nothing real there.")
print(f"   Fraction 'treated' by this tree: {frac_treated_const:.2%}")
print(f"   Truth: should be 100% (positive constant effect = treat everyone)")

### 🚨 Lesson from the Failure Case

**The tree will always split if you let it.** Greedy tree learners find *some* splits even when there's no real heterogeneity to find. The fix is the same as in Lab 4:

1. **Run `test_calibration()`** (or its Python equivalent) first to confirm real heterogeneity exists
2. **Compare policy values** with confidence intervals — if the tree's value isn't significantly better than treat-all, don't trust the structure
3. **Use honest estimation** (cross-fitting, which we did here) — it doesn't prevent spurious splits but it makes the noise unbiased

For PS 4, you'll run a similar diagnostic on the St. Null's data using `grf::test_calibration()`.

## 7. Budget Constraint: $\alpha = 30\%$

CEO Beta: *"Love the tree. But we can only afford to enroll 30% of patients. Who goes first?"*

**The optimal constrained rule** (top-$q$ targeting): rank by $\hat\tau$ and treat the top 30%.

**The interpretable constrained rule**: find the best depth-2 tree that treats no more than 30%.

Both use the same DR scores. Let's compare.

In [ ]:
alpha = 0.30
n = len(df)
n_budget = int(alpha * n)

# Top-q targeting
threshold = np.sort(tau_hat)[-n_budget]
pi_topq = (tau_hat >= threshold).astype(int)
v_topq, se_topq = policy_value(pi_topq, gamma_0, gamma_1)

# Budget-constrained depth-2 tree: Lagrangian approach
# Subtract a cost lambda from every treated action until the tree treats ~30%
# Binary-search over lambda for a depth-2 tree that honors the budget.
def fit_budgeted_tree(X, gamma_0, gamma_1, max_depth=2, budget=0.30, tol=0.02):
    lo, hi = 0.0, max(gamma_1 - gamma_0) + 0.1
    best = None
    for _ in range(30):
        lam = (lo + hi) / 2
        tree = fit_policy_tree(X, gamma_0, gamma_1 - lam, max_depth=max_depth)
        frac = tree.predict(X).mean()
        if frac > budget + tol:
            lo = lam  # penalize treatment more
        elif frac < budget - tol:
            hi = lam  # penalize treatment less
        else:
            best = tree
            break
        best = tree
    return best

tree_budget = fit_budgeted_tree(X, gamma_0, gamma_1, max_depth=2, budget=alpha)
pi_tree_budget = tree_budget.predict(X)
v_tree_budget, se_tree_budget = policy_value(pi_tree_budget, gamma_0, gamma_1)

# Plot comparison
fig, ax = plt.subplots(figsize=(11, 4.5))
plot_tree(tree_budget, feature_names=feature_names, class_names=["Don't","Treat"],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title(f"Depth-2 policy tree under 30% budget constraint  "
             f"(actual treated: {pi_tree_budget.mean():.1%})")
plt.tight_layout()
plt.show()

print(f"\n{'Policy':<32} {'Treated':<10} {'Value':<8} {'95% CI'}")
print("-" * 68)
for name, pi, v, se in [
    ("Unconstrained depth-2 tree", pi_tree2, v_tree2, se_tree2),
    (f"Top-{alpha:.0%} by τ̂", pi_topq, v_topq, se_topq),
    (f"Depth-2 tree, {alpha:.0%} budget", pi_tree_budget, v_tree_budget, se_tree_budget),
]:
    print(f"{name:<32} {pi.mean():>6.1%}    {v:>6.3f}   [{v-1.96*se:.2f}, {v+1.96*se:.2f}]")

### What the Budget Constraint Teaches You

Notice how the budgeted tree is often **simpler** than the unconstrained tree — maybe a single split. With less capacity, you can't afford to spend your budget on marginal-benefit patients, so the tree collapses to the "obviously high-benefit" group.

**Three ways the constrained tree can differ from top-$q$:**

1. **Interpretability floor** — top-$q$ might select a scattered 30% of the population; the tree forces a contiguous decision rule
2. **Regularization bonus** — the tree class resists overfitting; top-$q$ ranks on noisy $\hat\tau$
3. **Slightly lower value** — you pay for interpretability with a small value gap (the "regret of restriction")

This is the core tradeoff of the whole unit.

## 8. AI Guidance Notes

Some things to watch for when you're running policy learning on your own data — especially when using AI assistants:

### 🤖 When AI is helpful

- **Generating synthetic DGPs** for testing — ask: *"Generate a sklearn pipeline that creates n=1000 observations with known CATE structure, then fits a T-learner"*
- **Explaining outputs** — paste a `plot_tree` figure and ask what each split means
- **Debugging package errors** — `econml`, `causalml`, and `grf` have quirky APIs and version compatibility issues

### 🚩 When to be skeptical

- **AI will invent sklearn API calls that don't exist.** `DecisionTreeClassifier` has no `policy_value` method — I wrote one by hand above. Don't let an assistant make up functions.
- **AI often forgets cross-fitting.** It'll suggest fitting the outcome model on all data and predicting on all data — that's in-sample bias. Always check: was the model trained on the observations it's predicting on?
- **AI will recommend the first library it thinks of** (`econml`, `causalml`, `sklearn`, `grf` via `rpy2`) without understanding your context. In class, we used pure `sklearn` for transparency. For your research, pick based on what you can audit.

### 🧪 How to debug your own policy tree

1. **Always verify on synthetic data first.** If your tree doesn't recover a planted policy on a simulated DGP you control, don't trust it on real data.
2. **Check the treated fraction.** If a depth-2 tree treats 99% or 1% of the population, something is wrong — probably sign error in DR scores or the outcome being negative-is-good.
3. **Look at the leaves.** If any leaf has < 50 observations, reduce tree depth or increase `min_samples_leaf`.
4. **Compare to `treat-all`.** If the tree's DR value CI overlaps treat-all, you don't have enough heterogeneity to justify a tree.

### 📋 A pre-flight checklist for PS 4

Before you interpret any policy tree:

- [ ] Did you check calibration? (real heterogeneity?)
- [ ] Are DR scores using cross-fitted nuisances? (no in-sample bias)
- [ ] Did you verify with simulation first?
- [ ] Is every leaf $\geq$ 50 observations?
- [ ] Does the tree beat treat-all with non-overlapping CIs?
- [ ] Can a non-statistician read the tree?

## Wrap-Up

**What you've seen today:**

1. **Synthetic DGP** with known optimal policy — the gold standard for method validation
2. **Cross-fit T-learner** for honest CATE estimation
3. **Per-action DR scores** — the input to every policy learning algorithm
4. **Depth-2 policy tree** fit by weighted classification on DR scores
5. **Failure case** — trees split even when nothing's there; diagnose with value CIs
6. **Budget constraint** via Lagrangian penalty — the tree simplifies under tight capacity

**For Problem Set 4**, you'll do this workflow in **R** using `grf::causal_forest()`, `grf::double_robust_scores()`, and `policytree::policy_tree()` on the St. Null's dataset. The conceptual pipeline is identical — the syntax is different.

**For Wednesday (Jigsaw 4):** these same tools appear in all four papers, applied to Medicaid (Oregon HIE), anti-malarial subsidies (Kenya), depression treatment (dysthymic RCT), and fair targeting. Come ready to critique.

---

*Session 22 Code-Along — HPM 883 Spring 2026 — Sean Sylvia*